# Scenario Comparison with AMBER

Compare multiple economic scenarios side-by-side using AMBER's DataFrame-native results.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from climapan_lab.base_params import economic_params
from climapan_lab.src.models import EconModel
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

## 1. Define scenarios

In [ ]:
def run_scenario(name, overrides):
    p = economic_params.copy()
    p.update({
        'c_agents': 100, 'capitalists': 10, 'csf_agents': 3, 'cpf_agents': 2,
        'steps': 120, 'seed': 42, 'show_progress': False,
        'covid_settings': None, 'climateModuleFlag': False,
        'verboseFlag': False,
    })
    p.update(overrides)
    m = EconModel(p)
    result = m.run()
    df = result['model']
    if 'GDP' in df.columns:
        df = df.with_columns(pl.lit(name).alias('scenario'))
    return df

# Run 3 scenarios with different carbon tax rates
scenarios = {
    'BAU (no tax)':   {'fossilFuelPriceGrowth': 0.0},
    'Low carbon tax': {'fossilFuelPriceGrowth': 0.001},
    'High carbon tax':{'fossilFuelPriceGrowth': 0.005},
}

results = {}
for name, overrides in scenarios.items():
    print(f'Running {name}...')
    results[name] = run_scenario(name, overrides)
    print(f'  Done in {len(results[name])} records')

## 2. Combine and plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for name, df in results.items():
    if 'GDP' in df.columns:
        gdp = df['GDP'].drop_nulls().to_numpy()
        axes[0].plot(gdp, label=name, linewidth=1.5)
    if 'Gini' in df.columns:
        gini = df['Gini'].drop_nulls().to_numpy()
        axes[1].plot(gini, label=name, linewidth=1.5)
    if 'Investment' in df.columns:
        inv = df['Investment'].drop_nulls().to_numpy()
        axes[2].plot(inv, label=name, linewidth=1.5)

axes[0].set_title('GDP'); axes[0].set_xlabel('Month')
axes[1].set_title('Gini Coefficient'); axes[1].set_xlabel('Month')
axes[2].set_title('Investment'); axes[2].set_xlabel('Month')
axes[2].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Polars-native analysis of results

Since AMBER returns Polars DataFrames, you can chain operations efficiently.

In [ ]:
# Combine all scenarios into one DataFrame
all_dfs = [df for df in results.values() if df.height > 0]
if all_dfs:
    combined = pl.concat(all_dfs, how='vertical')
    
    # Polars-native summary: mean GDP and Gini per scenario
    summary = combined.group_by('scenario').agg([
        pl.col('GDP').mean().alias('avg_GDP'),
        pl.col('Gini').mean().alias('avg_Gini'),
        pl.col('GDP').std().alias('std_GDP'),
    ]).sort('scenario')
    
    print(summary)

## 4. AMBER Experiment API (parameter sweeps)

AMBER provides built-in experiment management for parameter exploration.

In [ ]:
import ambr as am

# Define parameter ranges to sweep
sample = am.Sample({
    'c_agents': [50, 100],
    'capitalists': am.IntRange(5, 15),
}, n=4)

print(f'Parameter combinations: {len(sample.combinations)}')
for i, combo in enumerate(sample.combinations):
    print(f'  {i+1}. agents={combo["c_agents"]}, capitalists={combo["capitalists"]}')

## 5. End-to-end experiment

Run all combinations and collect results in one DataFrame.

In [ ]:
all_model_dfs = []

for combo in sample.combinations:
    p = economic_params.copy()
    p.update({
        'c_agents': combo['c_agents'],
        'capitalists': combo['capitalists'],
        'csf_agents': 2, 'cpf_agents': 1,
        'steps': 60, 'seed': 42, 'show_progress': False,
        'covid_settings': None, 'climateModuleFlag': False,
    })
    m = EconModel(p)
    r = m.run()
    if r['model'].height > 0:
        tagged = r['model'].with_columns([
            pl.lit(combo['c_agents']).alias('n_agents'),
            pl.lit(combo['capitalists']).alias('n_capitalists'),
        ])
        all_model_dfs.append(tagged)

if all_model_dfs:
    experiment_df = pl.concat(all_model_dfs, how='vertical')
    print(f'Experiment results: {experiment_df.height} rows x {len(experiment_df.columns)} cols')
    
    # Average GDP by agent count
    gdp_by_n = experiment_df.group_by('n_agents').agg(
        pl.col('GDP').mean().alias('mean_GDP')
    ).sort('n_agents')
    print(gdp_by_n)